In [ ]:
# 16_constrained_coupled_dice.ipynb
# C-1: re-run DiCE with a coupled-feasibility filter -- waist reduction supported by
# weight loss, BMI consistent with weight/height, carbohydrate reduction <= 30%.
# Reports the share of DiCE counterfactuals that remain coherent AND risk-reducing,
# and the mean achievable risk reduction before vs after the filter.
# Runs locally/cloud; no API key (uses saved deploy models). Writes
# results/tables/table_c1_constrained_dice.csv. NOTE: DiCE random search is slow;
# N_SAMPLE is set to 40 per sex for a tractable demonstration (raise to reproduce fully).
import warnings, signal
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb, dice_ml
warnings.filterwarnings("ignore")
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA, MODELS, TAB = ROOT/"data", ROOT/"models", ROOT/"results"/"tables"
SEED=42; np.random.seed(SEED)
class _T(Exception): pass
signal.signal(signal.SIGALRM, lambda s,f:(_ for _ in ()).throw(_T()))
TIMEOUT=30
PRED=["age","HE_BMI","HE_wc","HE_wt","HE_ht","N_EN","N_CHO","N_SUGAR","N_CHOL","sm_presnt","dr_month","pa_aerobic","L_BR_FQ","edu","ho_incm","D_1_1"]
CATEG=["sm_presnt","dr_month","pa_aerobic"]; CONT=[c for c in PRED if c not in CATEG]
IMMUT=["age","HE_ht","edu","ho_incm","D_1_1"]; VARY=[c for c in PRED if c not in IMMUT]
WAIST_CAP=15.0; PCT={"HE_BMI":0.20,"HE_wt":0.20}; N_SAMPLE=40; TOTAL_CF=8
CARB_CAP=0.30; WAIST_PER_KG=1.0; SLACK=1.5; BMI_TOL=0.5
def coherent(xo,xcf):
    dwt=float(xo["HE_wt"]-xcf["HE_wt"]); dwc=float(xo["HE_wc"]-xcf["HE_wc"])
    if dwc > SLACK*WAIST_PER_KG*max(dwt,0)+2.0: return False
    if abs(float(xcf["HE_wt"])/((float(xcf["HE_ht"])/100)**2)-float(xcf["HE_BMI"]))>BMI_TOL: return False
    if float(xo["N_CHO"])>0 and (float(xo["N_CHO"])-float(xcf["N_CHO"]))/float(xo["N_CHO"])>CARB_CAP: return False
    return True
rows=[]
for sx in ("Male","Female"):
    full=pd.read_parquet(DATA/f"scored_{sx.lower()}.parquet"); data_df=full[PRED+["diabetes"]].copy()
    for c in CATEG: data_df[c]=data_df[c].astype(int).astype(str)
    deploy=xgb.XGBClassifier(); deploy.load_model(str(MODELS/f"xgb_{sx.lower()}_deploy.json"))
    class Wrap:
        def __init__(s,m): s.m=m
        def predict_proba(s,X):
            Z=X.copy()
            for c in CATEG:
                if Z[c].dtype==object: Z[c]=Z[c].astype(float)
            return s.m.predict_proba(Z[PRED].astype(float))
        def predict(s,X): return (s.predict_proba(X)[:,1]>=0.5).astype(int)
    d=dice_ml.Data(dataframe=data_df,continuous_features=CONT,outcome_name="diabetes")
    m=dice_ml.Model(model=Wrap(deploy),backend="sklearn",model_type="classifier")
    exp=dice_ml.Dice(d,m,method="random")
    hr=pd.read_parquet(DATA/f"highrisk_nondiab_{sx.lower()}.parquet").sample(min(N_SAMPLE,10**9),random_state=SEED).reset_index(drop=True) if False else pd.read_parquet(DATA/f"highrisk_nondiab_{sx.lower()}.parquet").sample(N_SAMPLE,random_state=SEED).reset_index(drop=True)
    n_any=n_feas=0; ru=[]; rc=[]
    for i in range(len(hr)):
        x=hr.iloc[i]; row=hr.iloc[[i]][PRED].copy()
        for c in CATEG: row[c]=row[c].astype(int).astype(str)
        pr={"HE_wc":[float(x.HE_wc-WAIST_CAP),float(x.HE_wc)]}
        for v,cap in PCT.items(): pr[v]=[float(x[v]*(1-cap)),float(x[v])]
        try:
            signal.alarm(TIMEOUT)
            cf=exp.generate_counterfactuals(row,total_CFs=TOTAL_CF,desired_class=0,features_to_vary=VARY,permitted_range=pr,random_seed=SEED)
            signal.alarm(0); cfdf=cf.cf_examples_list[0].final_cfs_df
            if cfdf is None or len(cfdf)==0: continue
            Xcf=cfdf[PRED].astype(float); probs=deploy.predict_proba(Xcf)[:,1]
            op=float(deploy.predict_proba(hr.iloc[[i]][PRED].astype(float))[:,1][0]); n_any+=1
            ru.append(op-probs[int(np.argmin(probs))])
            feas=[k for k in range(len(Xcf)) if coherent(x,Xcf.iloc[k]) and probs[k]<op]
            if feas: n_feas+=1; rc.append(op-min(probs[k] for k in feas))
        except (_T,Exception):
            signal.alarm(0); continue
    rows.append([sx,len(hr),n_any,n_feas,round(np.mean(ru),3) if ru else np.nan,round(np.mean(rc),3) if rc else np.nan])
    print(f"{sx}: any_CF={n_any} coherent_feasible={n_feas} mean_red uncon={np.mean(ru) if ru else np.nan:.3f} con={np.mean(rc) if rc else np.nan:.3f}")
pd.DataFrame(rows,columns=["Sex","Sampled","Any_CF","Coherent_feasible_CF","Mean_red_unconstrained","Mean_red_constrained"]).to_csv(TAB/"table_c1_constrained_dice.csv",index=False)
print("C-1 constrained DiCE written")
